# SBN Completed Inspections — Daily Refresh (AIL-1683)

Pulls completed inspections directly from the SBN API and **upserts** them into
`SBN | Completed Inspections | RAW`, replacing the unreliable emailed
"Completed Inspections by Checklist" report that randomly drops rows.

Each run re-pulls a rolling window and merges on inspection `id`, so any row a
previous run missed is **self-healed** on the next run, and history older than
the window is preserved untouched.

- **Plan:** `docs/plans/2026-06-22-feat-sbn-completed-inspections-api-notebook-plan.md`
- **Linear:** [AIL-1683](https://linear.app/wksusa/issue/AIL-1683)
- **One-time seed:** run `SBN_Completed_Inspections_Backfill.ipynb` once before enabling this schedule.

Runs **inside the Domo Jupyter Workspace** on a daily schedule (set in the
Workspace UI, early AM). Shares `sbn_common.py` + `sbn.env` with the backfill notebook.


In [11]:
# Bootstrap: make sbn_common importable, load credentials from sbn.env.
import os, sys
from pathlib import Path
from datetime import date, timedelta

sys.path.insert(0, str(Path.cwd()))  # Workspace runs notebooks from their folder
from dotenv import load_dotenv
import sbn_common as sc

ENV_FILE = "sbn.env"
load_dotenv(ENV_FILE)
# --- Snowflake port: no sbn.env in Snowflake; load the same two values from a Snowflake secret ---
from snowflake.snowpark.secrets import get_username_password
_sbn = get_username_password(os.environ["SBN_SECRET_PATH"])  # set in code_bundle.yml
os.environ.setdefault("SBN_USERNAME", _sbn.username)
os.environ.setdefault("SBN_PASSWORD", _sbn.password)
os.chdir("/mnt/sbn_state")  # Code Bundle folder is read-only; work from a persistent stage so ./state survives runs
# --- end Snowflake port ---
SBN_USERNAME = os.getenv("SBN_USERNAME")
SBN_PASSWORD = os.getenv("SBN_PASSWORD")
BASE_URL = os.getenv("SBN_BASE_URL", sc.DEFAULT_BASE_URL)
if not SBN_USERNAME or not SBN_PASSWORD:
    raise ValueError(f"SBN_USERNAME / SBN_PASSWORD missing from {ENV_FILE}")

DATASET_NAME = "SBN | Completed Inspections | RAW"
# DATASET_NAME = "SBN_COMPLETED_INSPECTIONS_RAW"  # Snowflake-friendly name: comment out the line above and uncomment this one; also change the table name in backfill_seed.py, 01b_create_table.sql, 03_initial_load.sql and 04_schedule_and_monitor.sql
PAGE_SIZE = sc.DEFAULT_PAGE_SIZE
print("✓ Bootstrap ready — base:", BASE_URL, "| dataset:", DATASET_NAME)


✓ Bootstrap ready — base: https://wks.simplebutneeded.net/api/v1 | dataset: SBN | Completed Inspections | RAW


In [12]:
# Authenticate + build the (small) checklist & user lookup maps.
# A closure lets sbn_get re-login once if the session expires mid-run.
session = sc.make_session()

def _login():
    sc.login(session, BASE_URL, SBN_USERNAME, SBN_PASSWORD)

_login()
relogin = _login

checklist_map = sc.build_lookup_map(session, BASE_URL, "checklist", lambda r: r.get("name"), relogin=relogin)
user_map = sc.build_lookup_map(
    session, BASE_URL, "user",
    lambda r: (r.get("first_name") or "", r.get("last_name") or ""),
    core=True, relogin=relogin,
)
print(f"✓ {len(checklist_map)} checklists, {len(user_map)} users")


2026-06-23 00:03:54,309 INFO Authenticated to SBN as grace.brockwell@wksusa.com
2026-06-23 00:03:54,699 INFO Built checklist lookup map: 107 entries
2026-06-23 00:03:54,993 INFO Built user lookup map: 510 entries


✓ 107 checklists, 510 users


In [13]:
# Daily window parameters.
ROLLING_DAYS = 30
window_lo = sc.iso_utc(date.today() - timedelta(days=ROLLING_DAYS))
sc.log.info("Rolling window: completion_date >= %s (last %d days)", window_lo, ROLLING_DAYS)


2026-06-23 00:03:54,998 INFO Rolling window: completion_date >= 2026-05-24T00:00:00+00:00 (last 30 days)


In [14]:
# Fetch the union of (completed in window) and (modified in window).
#  - completed-window  -> self-heals rows the email dropped (count-gated).
#  - modified-window   -> self-heals edits AND surfaces status reversions.
completed = sc.fetch_completed_jobs(session, BASE_URL, completion_gte=window_lo,
                                    page_size=PAGE_SIZE, relogin=relogin)
modified = sc.fetch_recently_modified(session, BASE_URL, modified_gte=window_lo,
                                      page_size=PAGE_SIZE, relogin=relogin)

# Union by id; completed records are authoritative over the modified-window copy.
by_id = {j["id"]: j for j in modified}
by_id.update({j["id"]: j for j in completed})
completed_jobs = [j for j in by_id.values() if j.get("job_status") == "completed"]

# Reversion visibility: jobs touched in-window that are NOT completed. We don't
# write them as completed; acting on removals is out of scope for v1 (a prior
# completed row for these ids lingers in the dataset — documented in README).
reverted = [j["id"] for j in by_id.values() if j.get("job_status") != "completed"]
if reverted:
    sc.log.warning("%d in-window jobs are not 'completed' (edits/reversions); first 20 ids: %s",
                   len(reverted), reverted[:20])

fresh = sc.build_report_frame(completed_jobs, checklist_map, user_map)
sc.log.info("Built %d completed report rows", len(fresh))
print(f"✓ fresh rows: {len(fresh):,}")  # count only — never print raw PII rows


2026-06-23 00:03:56,503 INFO Fetched 1237 completed jobs (API count before=1237 after=1237)
2026-06-23 00:03:57,188 INFO Fetched 1495 recently-modified jobs (modified >= 2026-05-24T00:00:00+00:00)
2026-06-23 00:03:57,191 WARNING 258 in-window jobs are not 'completed' (edits/reversions); first 20 ids: [4902956, 4902958, 4902961, 4902967, 4902969, 4902970, 4902986, 4902987, 4903059, 4903084, 4903098, 4903148, 4903387, 4903396, 4903483, 4903487, 4903706, 4903707, 4903708, 4903709]
2026-06-23 00:03:57,214 INFO Built 1237 completed report rows


✓ fresh rows: 1,237


In [15]:
# Read existing -> upsert on id -> REPLACE. Tri-state read safety: we ABORT
# rather than risk overwriting good history with a partial pull.
import domojupyter as domo

STATE = Path("./state"); STATE.mkdir(exist_ok=True)
COUNT_FILE = STATE / "last_row_count.txt"
last_known = int(COUNT_FILE.read_text().strip()) if COUNT_FILE.exists() else None

try:
    existing = domo.read_dataframe(DATASET_NAME)
except Exception as e:
    raise RuntimeError(
        f"Could not read existing '{DATASET_NAME}'. Aborting so a partial pull never "
        f"overwrites good history. If this is the FIRST run, create the dataset as a "
        f"Workspace Output and seed it with the Backfill notebook first. Cause: {e}"
    )

if len(existing) == 0 and last_known and last_known > 0:
    raise RuntimeError(
        f"Existing dataset read returned 0 rows but the last run wrote {last_known}. "
        f"Treating as a transient read failure and aborting without write."
    )

merged = sc.upsert_on_id(existing, fresh)
sc.log.info("Upsert: existing=%d + fresh=%d -> %d rows", len(existing), len(fresh), len(merged))
domo.write_dataframe(merged, DATASET_NAME, update_method="REPLACE")
COUNT_FILE.write_text(str(len(merged)))
print(f"✓ wrote {len(merged):,} rows to '{DATASET_NAME}'")


2026-06-23 00:04:00,141 INFO Upsert: existing=26778 + fresh=1237 -> 26778 rows


✓ wrote 26,778 rows to 'SBN | Completed Inspections | RAW'


In [16]:
# Validation summary (aggregates only — no raw rows in output).
print(f"Total rows:            {len(merged):,}")
print(f"Distinct inspections:  {merged['id'].nunique():,}")
print(f"Completion date range: {merged['Completion Date'].min()} -> {merged['Completion Date'].max()}")
print(f"Distinct checklists:   {merged['Checklist Name'].nunique()}")
print(f"Distinct Lv1 (brand):  {sorted(merged['Location Lv1'].dropna().unique())}")
unresolved = int(merged['Checklist Name'].str.startswith('checklist#').sum())
if unresolved:
    print(f"⚠ {unresolved} rows have an unresolved checklist id (rendered as checklist#<id>)")


Total rows:            26,778
Distinct inspections:  26,778
Completion date range: 2024-12-31 16:02:21 -> 2026-06-22 21:10:18.477000
Distinct checklists:   21
Distinct Lv1 (brand):  ['CARWOOD CARWASH', "Denny's", 'El Pollo Loco', 'FACILITIES DEPARMENT', 'Krispy Kreme', 'SUPER 8 HOTEL', 'SUPPORT CENTER', "Wendy's", 'ZZZ Web Form']
